# Marketing Campaign Analyzer - CSV Parser Walkthrough

This notebook exercises `services/csv_parser.py` - the first layer of the
`POST /api/analyze` endpoint. Before any AI call happens, `parse_csv()` is
responsible for reading the uploaded file, detecting the platform, computing
all derived metrics (CTR, CPC, CPA, ROAS, Budget Efficiency), and building
the payload that gets forwarded to `ai_service.analyse_campaigns()`.

In [ ]:
import io, re, json
from pathlib import Path
from typing import Optional
import pandas as pd

DATA_DIR = Path().resolve().parent / 'data'
META_FILE = DATA_DIR / 'meta_export.csv'
GOOGLE_FILE = DATA_DIR / 'google_export.csv'

for f in [META_FILE, GOOGLE_FILE]:
    assert f.exists(), f'Missing: {f} - make sure data/ folder is next to this notebook'
    print(f'  {f}  ({f.stat().st_size:,} bytes)')


  D:\Marketing Analyzer\backend\data\meta_export.csv  (791 bytes)
  D:\Marketing Analyzer\backend\data\google_export.csv  (882 bytes)


---
## 1. Parser internals - `services/csv_parser.py`

These are the exact functions from the production file, copied here so the
notebook runs without importing FastAPI.


In [ ]:
META_COLUMNS = {
    "campaign_name": ["Campaign name", "Campaign Name"],
    "impressions": ["Impressions"],
    "clicks": ["Clicks (all)", "Clicks"],
    "spend": ["Amount spent (EUR)", "Amount spent (USD)", "Amount spent"],
    "ctr": ["CTR (all)", "CTR"],
    "cpc": ["CPC (all)", "CPC (cost per link click)", "CPC"],
    "conversions": ["Results", "Conversions"],
    "cpa": ["Cost per result", "Cost per conversion"],
    "revenue": ["Purchase conversion value", "Conv. value"],
}
GOOGLE_COLUMNS = {
    "campaign_name": ["Campaign"],
    "impressions": ["Impr.", "Impressions"],
    "clicks": ["Clicks"],
    "spend": ["Cost", "Spend"],
    "ctr": ["CTR"],
    "cpc": ["Avg. CPC"],
    "conversions": ["Conversions"],
    "cpa": ["Cost / conv.", "CPA"],
    "revenue": ["Conv. value", "Conversion value", "All conv. value"],
}


def _detect_platform(df: pd.DataFrame) -> str:
    cols = set(df.columns.str.lower())
    if any("amount spent" in c for c in cols): return "meta"
    if any("avg. cpc" in c for c in cols) or "impr." in cols: return "google"
    return "unknown"


def _find_column(df: pd.DataFrame, candidates: list) -> Optional[str]:
    for c in candidates:
        if c in df.columns: return c
    lower_map = {col.lower(): col for col in df.columns}
    for c in candidates:
        if c.lower() in lower_map: return lower_map[c.lower()]
    return None


def _safe_float(val) -> float:
    if pd.isna(val): return 0.0
    s = str(val).strip()
    if not s or s == "--": return 0.0
    s = re.sub(r'[^\d,.\-]', '', s)
    if ',' in s and '.' in s and s.rfind(',') > s.rfind('.'):
        s = s.replace('.', '').replace(',', '.')
    else:
        s = s.replace(',', '')
    try: return float(s)
    except ValueError: return 0.0


def _calculate_efficiency(row: dict) -> float:
    """Budget Efficiency Score 0-100: ROAS ≤40 pts, CTR ≤30 pts, CPA ≤30 pts."""
    roas_score = min(row["roas"] / 5 * 40, 40)
    ctr_score = min(row["ctr"] / 3 * 30, 30)
    cpa_score = max(0, 30 - (row["cpa"] / 50 * 30))
    return round(roas_score + ctr_score + cpa_score, 1)


def _compute_roas(spend: float, revenue: float, conv: float,
                  avg_order_value: float = None) -> tuple:
    """Mirrors the updated _compute_roas() in csv_parser.py.
    Priority: actual revenue col > avg_order_value param > unavailable.
    Returns (roas_value, method_used)."""
    if spend <= 0: return 0.0, 'unavailable'
    if revenue > 0: return round(revenue / spend, 2), 'actual_revenue'
    if avg_order_value and conv > 0:
        return round((conv * avg_order_value) / spend, 2), 'avg_order_value'
    return 0.0, 'unavailable'


def parse_csv(file_bytes: bytes, avg_order_value: float = None) -> dict:
    """Mirrors parse_csv() from services/csv_parser.py (no FastAPI dependency)."""
    df = pd.read_csv(io.BytesIO(file_bytes)).dropna(how='all')
    platform = _detect_platform(df)
    col_map  = META_COLUMNS if platform == 'meta' else GOOGLE_COLUMNS

    name_col = _find_column(df, col_map['campaign_name'])
    imp_col = _find_column(df, col_map['impressions'])
    clk_col = _find_column(df, col_map['clicks'])
    spd_col = _find_column(df, col_map['spend'])
    conv_col = _find_column(df, col_map['conversions'])
    rev_col = _find_column(df, col_map.get('revenue', []))

    if not all([name_col, imp_col, clk_col, spd_col]):
        raise ValueError('CSV missing required columns — name, impressions, clicks or spend')

    campaigns = []
    for _, row in df.iterrows():
        name = str(row.get(name_col, '')).strip()
        if not name: continue
        imp = int(_safe_float(row.get(imp_col, 0)))
        clk = int(_safe_float(row.get(clk_col, 0)))
        spd = _safe_float(row.get(spd_col,  0))
        conv = _safe_float(row.get(conv_col, 0)) if conv_col else 0.0
        rev = _safe_float(row.get(rev_col, 0)) if rev_col  else 0.0

        ctr = (clk / imp * 100) if imp  > 0 else 0.0
        cpc = (spd / clk) if clk  > 0 else 0.0
        cpa = (spd / conv) if conv > 0 else 0.0
        roas, roas_method = _compute_roas(spd, rev, conv, avg_order_value)

        campaigns.append({
            'campaign_name': name,
            'impressions': imp,
            'clicks': clk,
            'spend': round(spd, 2),
            'conversions': round(conv, 2),
            'ctr': round(ctr, 2),
            'cpc': round(cpc, 2),
            'cpa': round(cpa, 2),
            'roas': roas,
            'roas_method': roas_method,
            'budget_efficiency': _calculate_efficiency({'roas': roas, 'ctr': ctr, 'cpa': cpa}),
        })

    if not campaigns:
        raise ValueError('No valid campaign rows found in CSV')

    total_spend = round(sum(c['spend'] for c in campaigns), 2)
    total_clicks = sum(c['clicks'] for c in campaigns)
    total_imps = sum(c['impressions'] for c in campaigns)
    total_conv = round(sum(c['conversions'] for c in campaigns), 2)
    avg_ctr = round(total_clicks / total_imps * 100, 2) if total_imps > 0 else 0
    avg_cpc = round(total_spend  / total_clicks, 2) if total_clicks > 0 else 0
    avg_roas = round(sum(c['roas'] for c in campaigns) / len(campaigns), 2)

    sorted_roas = sorted(campaigns, key=lambda x: x['roas'], reverse=True)
    top_performers = [c['campaign_name'] for c in sorted_roas[:3]]
    bottom_perfs = [c['campaign_name'] for c in sorted_roas[-3:]]

    return {
        'platform': platform,
        'total_spend': total_spend,
        'total_clicks': total_clicks,
        'total_impressions': total_imps,
        'total_conversions': total_conv,
        'avg_ctr': avg_ctr,
        'avg_cpc': avg_cpc,
        'avg_roas': avg_roas,
        'campaigns': campaigns,
        'top_performers': top_performers,
        'bottom_performers': bottom_perfs,
        'metrics_json': json.dumps(campaigns, indent=2),
    }

print('Parser functions loaded - identical to services/csv_parser.py')


Parser functions loaded - identical to services/csv_parser.py


---
## 2. Parse `meta_export.csv`

This is what happens when a user uploads their Meta Ads CSV to `POST /api/analyze`.
The file has no `Conv. value` column, so ROAS will be `unavailable` until the
user provides `avg_order_value` as a query param.


In [ ]:
meta_bytes = META_FILE.read_bytes()
meta_parsed = parse_csv(meta_bytes)

print(f'Platform: {meta_parsed["platform"]}')
print(f'Campaigns parsed: {len(meta_parsed["campaigns"])}')
print(f'Total spend: €{meta_parsed["total_spend"]:,.2f}')
print(f'Total conversions: {meta_parsed["total_conversions"]:.0f}')
print(f'Avg CTR: {meta_parsed["avg_ctr"]}%')
print(f'Avg CPC: €{meta_parsed["avg_cpc"]}')
print(f'Avg ROAS: {meta_parsed["avg_roas"]}  '
      f'({"unavailable - no conv. value column" if meta_parsed["avg_roas"] == 0 else "ok"})')


Platform: meta
Campaigns parsed: 10
Total spend: €43,030.50
Total conversions: 856
Avg CTR: 2.0%
Avg CPC: €1.21
Avg ROAS: 0.0  (unavailable - no conv. value column)


### 2a. Campaign-level metrics table

This is the table shown in the frontend dashboard after a successful `/api/analyze` call.
The `efficiency` column is the composite score used to rank `top_performers` and `bottom_performers`.


In [ ]:
rows = meta_parsed['campaigns']
df_meta = pd.DataFrame(rows)

display = df_meta[['campaign_name','spend','impressions','clicks','conversions','ctr','cpc','cpa','budget_efficiency']].copy()
display.columns = ['Campaign','Spend €','Impressions','Clicks','Conv.','CTR %','CPC €','CPA €','Efficiency']
display = display.sort_values('Efficiency', ascending=False).reset_index(drop=True)

# Format for readability
display['Spend €'] = display['Spend €'].map('{:,.2f}'.format)
display['Impressions'] = display['Impressions'].map('{:,}'.format)
display['Clicks'] = display['Clicks'].map('{:,}'.format)
display['CTR %'] = display['CTR %'].map('{:.2f}'.format)
display['CPC €'] = display['CPC €'].map('{:.2f}'.format)
display['CPA €'] = display['CPA €'].map(lambda x: f'{x:,.2f}' if x > 0 else '-')
display['Efficiency'] = display['Efficiency'].map('{:.1f}'.format)

print(display.to_string(index=False))


                             Campaign  Spend € Impressions Clicks  Conv. CTR % CPC €  CPA € Efficiency
Dynamic Product Ads - Cart Abandoners 3,740.00      62,300  5,480  231.0  8.80  0.68  16.19       50.3
           Black Friday - Retargeting 4,120.00      48,200  3,910  198.0  8.11  1.05  20.81       47.5
            Win-Back - 180 Day Lapsed 1,980.00      34,100  2,890   76.0  8.48  0.69  26.05       44.4
              Brand Awareness - Video 2,980.00     520,000  4,100    0.0  0.79  0.73      -       37.9
              Summer Sale - Catalogue 4,890.00     143,900  3,270   89.0  2.27  1.50  54.94       22.7
           Black Friday - Prospecting 9,840.50     312,400  6,820  142.0  2.18  1.44  69.30       21.8
   Competitor Conquest - Search Style 4,650.00      97,800  1,820   22.0  1.86  2.55 211.36       18.6
     Spring Collection - Lookalike 3% 3,980.00     185,600  2,740   38.0  1.48  1.45 104.74       14.8
         Spring Collection - Broad EU 5,210.00     290,100  3,620   51.0 

### 2b. Top & bottom performers - as returned by `AnalysisResponse`


In [ ]:
print('top_performers (highest ROAS):')
for name in meta_parsed['top_performers']:
    c = next(x for x in rows if x['campaign_name'] == name)
    print(f'  {name:<45}  CTR={c["ctr"]}%  CPA=€{c["cpa"]:,.2f}  efficiency={c["budget_efficiency"]}')

print()
print('bottom_performers (lowest ROAS):')
for name in meta_parsed['bottom_performers']:
    c = next(x for x in rows if x['campaign_name'] == name)
    print(f'  {name:<45}  CTR={c["ctr"]}%  CPA=€{c["cpa"]:,.2f}  efficiency={c["budget_efficiency"]}')

# Campaigns with zero conversions - the AI will flag these
zero = [c for c in rows if c['conversions'] == 0]
if zero:
    print()
    print('Zero-conversion campaigns (budget waste risk):')
    for c in zero:
        print(f'  {c["campaign_name"]}  —  €{c["spend"]:,.2f} spent, 0 results')


top_performers (highest ROAS):
  Black Friday - Prospecting                     CTR=2.18%  CPA=€69.30  efficiency=21.8
  Black Friday - Retargeting                     CTR=8.11%  CPA=€20.81  efficiency=47.5
  Spring Collection - Lookalike 3%               CTR=1.48%  CPA=€104.74  efficiency=14.8

bottom_performers (lowest ROAS):
  Summer Sale - Catalogue                        CTR=2.27%  CPA=€54.94  efficiency=22.7
  Influencer Collab - Stories                    CTR=1.1%  CPA=€182.22  efficiency=11.0
  Win-Back - 180 Day Lapsed                      CTR=8.48%  CPA=€26.05  efficiency=44.4

Zero-conversion campaigns (budget waste risk):
  Brand Awareness - Video  —  €2,980.00 spent, 0 results


### 2c. ROAS - re-parsing with `avg_order_value`

When the user provides `?avg_order_value=80` in the API call, ROAS becomes
computable. This is the `avg_order_value` param added to `routers/analyze.py`.


In [ ]:
meta_with_aov = parse_csv(meta_bytes, avg_order_value=80.0)

print('ROAS per campaign (avg_order_value = €80):')
print(f'{"Campaign":<45} {"Spend":>8}  {"Conv.":>6}  {"ROAS":>6}  {"Method"}')
print('-' * 80)
for c in sorted(meta_with_aov['campaigns'], key=lambda x: x['roas'], reverse=True):
    roas_str = f'{c["roas"]:.2f}x' if c['roas'] > 0 else '-'
    print(f'{c["campaign_name"]:<45} €{c["spend"]:>7,.2f}  {c["conversions"]:>6.0f}  {roas_str:>6}  {c["roas_method"]}')

print(f'\nAvg ROAS account-wide: {meta_with_aov["avg_roas"]}x')


ROAS per campaign (avg_order_value = €80):
Campaign                                         Spend   Conv.    ROAS  Method
--------------------------------------------------------------------------------
Dynamic Product Ads - Cart Abandoners         €3,740.00     231   4.94x  avg_order_value
Black Friday - Retargeting                    €4,120.00     198   3.84x  avg_order_value
Win-Back - 180 Day Lapsed                     €1,980.00      76   3.07x  avg_order_value
Summer Sale - Catalogue                       €4,890.00      89   1.46x  avg_order_value
Black Friday - Prospecting                    €9,840.50     142   1.15x  avg_order_value
Spring Collection - Broad EU                  €5,210.00      51   0.78x  avg_order_value
Spring Collection - Lookalike 3%              €3,980.00      38   0.76x  avg_order_value
Influencer Collab - Stories                   €1,640.00       9   0.44x  avg_order_value
Competitor Conquest - Search Style            €4,650.00      22   0.38x  avg_order_va

---
## 3. Parse `google_export.csv`

Google's export includes a `Conv. value` column, so ROAS is calculated from
**actual revenue data** - no `avg_order_value` needed.


In [ ]:
google_bytes  = GOOGLE_FILE.read_bytes()
google_parsed = parse_csv(google_bytes)

print(f'Platform: {google_parsed["platform"]}')
print(f'Campaigns parsed: {len(google_parsed["campaigns"])}')
print(f'Total spend: €{google_parsed["total_spend"]:,.2f}')
print(f'Total conversions: {google_parsed["total_conversions"]:.0f}')
print(f'Avg ROAS: {google_parsed["avg_roas"]}x  (from actual Conv. value)')
print(f'Avg CPC: €{google_parsed["avg_cpc"]}')


Platform: google
Campaigns parsed: 10
Total spend: €52,480.00
Total conversions: 1193
Avg ROAS: 2.72x  (from actual Conv. value)
Avg CPC: €1.36


### 3a. Campaign breakdown with real ROAS


In [ ]:
g_rows = google_parsed['campaigns']
df_google = pd.DataFrame(g_rows)

display_g = df_google[['campaign_name','spend','conversions','ctr','cpa','roas','budget_efficiency','roas_method']].copy()
display_g.columns = ['Campaign','Spend €','Conv.','CTR %','CPA €','ROAS','Efficiency','ROAS source']
display_g = display_g.sort_values('ROAS', ascending=False).reset_index(drop=True)

display_g['Spend €'] = display_g['Spend €'].map('{:,.2f}'.format)
display_g['CTR %'] = display_g['CTR %'].map('{:.2f}'.format)
display_g['CPA €'] = display_g['CPA €'].map(lambda x: f'{x:,.2f}' if x > 0 else '—')
display_g['ROAS'] = display_g['ROAS'].map(lambda x: f'{x:.2f}x' if x > 0 else '—')
display_g['Efficiency'] = display_g['Efficiency'].map('{:.1f}'.format)

print(display_g.to_string(index=False))


                 Campaign   Spend €  Conv. CTR %  CPA €  ROAS Efficiency    ROAS source
     Brand Search - Exact  3,120.00  312.0 14.19  10.00 8.00x       94.0 actual_revenue
   RLSA - Cart Abandoners  2,140.00  168.0 13.80  12.74 7.85x       92.4 actual_revenue
     Brand Search - Broad  2,840.00  198.0  6.89  14.34 5.58x       91.4 actual_revenue
Performance Max - EU Shop 12,400.00  284.0  2.17  43.66 2.29x       43.8 actual_revenue
    Display - Remarketing  1,980.00   24.0  0.42  82.50 0.97x       11.9 actual_revenue
  Generic - Running Shoes  8,940.00   87.0  1.78 102.76 0.78x       24.0 actual_revenue
 Generic - Sports Apparel  7,620.00   61.0  1.27 124.92 0.64x       17.8 actual_revenue
        Competitor - Nike  5,820.00   28.0  1.77 207.86 0.38x       20.8 actual_revenue
    YouTube - Prospecting  2,640.00   12.0  0.20 220.00 0.36x        4.9 actual_revenue
      Competitor - Adidas  4,980.00   19.0  1.78 262.11 0.31x       20.3 actual_revenue


### 3b. Budget waste candidates - ROAS < 1.0x


In [ ]:
waste = [c for c in g_rows if 0 < c['roas'] < 1.0]
if waste:
    print('Campaigns spending more than they earn (ROAS < 1.0x):')
    for c in sorted(waste, key=lambda x: x['spend'], reverse=True):
        print(f'  {c["campaign_name"]:<45}  spend=€{c["spend"]:,.2f}  ROAS={c["roas"]:.2f}x')
else:
    print('No campaigns with ROAS < 1.0x')

print()
print('top_performers:', google_parsed['top_performers'])
print('bottom_performers:', google_parsed['bottom_performers'])


Campaigns spending more than they earn (ROAS < 1.0x):
  Generic - Running Shoes                        spend=€8,940.00  ROAS=0.78x
  Generic - Sports Apparel                       spend=€7,620.00  ROAS=0.64x
  Competitor - Nike                              spend=€5,820.00  ROAS=0.38x
  Competitor - Adidas                            spend=€4,980.00  ROAS=0.31x
  YouTube - Prospecting                          spend=€2,640.00  ROAS=0.36x
  Display - Remarketing                          spend=€1,980.00  ROAS=0.97x

top_performers: ['Brand Search - Exact', 'RLSA - Cart Abandoners', 'Brand Search - Broad']
bottom_performers: ['Competitor - Nike', 'YouTube - Prospecting', 'Competitor - Adidas']


---
## 4. What the AI receives - `metrics_json` payload

This is the exact string passed to `ai_service.analyse_campaigns()` in `ai_service.py`.
Seeing it here tells you whether the data is clean and complete before any token is spent.


In [ ]:
print('=== Meta payload (first 2 campaigns) ===')
first_two = json.loads(meta_with_aov['metrics_json'])[:2]
print(json.dumps(first_two, indent=2))

print()
print('=== Google payload (first 2 campaigns) ===')
first_two_g = json.loads(google_parsed['metrics_json'])[:2]
print(json.dumps(first_two_g, indent=2))

print()
print(f'Full Meta payload: {len(meta_with_aov["metrics_json"]):,} chars  '
      f'(~{len(meta_with_aov["metrics_json"]) // 4:,} tokens)')
print(f'Full Google payload: {len(google_parsed["metrics_json"]):,} chars  '
      f'(~{len(google_parsed["metrics_json"]) // 4:,} tokens)')
print()
print('Ready for Notebook 2 - AI prompt evaluation with this data.')


=== Meta payload (first 2 campaigns) ===
[
  {
    "campaign_name": "Black Friday - Prospecting",
    "impressions": 312400,
    "clicks": 6820,
    "spend": 9840.5,
    "conversions": 142.0,
    "ctr": 2.18,
    "cpc": 1.44,
    "cpa": 69.3,
    "roas": 1.15,
    "roas_method": "avg_order_value",
    "budget_efficiency": 31.0
  },
  {
    "campaign_name": "Black Friday - Retargeting",
    "impressions": 48200,
    "clicks": 3910,
    "spend": 4120.0,
    "conversions": 198.0,
    "ctr": 8.11,
    "cpc": 1.05,
    "cpa": 20.81,
    "roas": 3.84,
    "roas_method": "avg_order_value",
    "budget_efficiency": 78.2
  }
]

=== Google payload (first 2 campaigns) ===
[
  {
    "campaign_name": "Brand Search - Exact",
    "impressions": 48200,
    "clicks": 6840,
    "spend": 3120.0,
    "conversions": 312.0,
    "ctr": 14.19,
    "cpc": 0.46,
    "cpa": 10.0,
    "roas": 8.0,
    "roas_method": "actual_revenue",
    "budget_efficiency": 94.0
  },
  {
    "campaign_name": "Brand Search - Broa

---
## 5. Parser robustness - edge cases from real-world exports

Meta and Google exports are inconsistent across regions and account settings.
These are the failure modes `_safe_float()` and `_find_column()` must handle.


In [ ]:
print('_safe_float() - input formats:')
print(f'{"Input":<25} {"Output":>10}  Notes')
print('-' * 55)
test_vals = [
    ('1234.56', 1234.56, 'standard'),
    ('€1,234.56', 1234.56, 'euro symbol, EN thousands'),
    ('1.234,56', 1234.56, 'EU decimal format (DE/FR/BG)'),
    ('2.13%', 2.13, 'CTR as percentage string'),
    ('--', 0.0, 'Meta missing-data marker'),
    ('', 0.0, 'empty cell'),
    (None, 0.0, 'NaN / None'),
    ('1,000,000', 1000000, 'million impressions'),
]
all_pass = True
for val, expected, note in test_vals:
    result = _safe_float(val)
    ok = abs(result - expected) < 0.001
    all_pass = all_pass and ok
    print(f'{repr(val):<25} {result:>10}  {"✅" if ok else "❌"} {note}')

print()
print(f'All cases: {"✅ PASS" if all_pass else "❌ FAILURES — fix _safe_float before deploying"}')


_safe_float() - input formats:
Input                         Output  Notes
-------------------------------------------------------
'1234.56'                    1234.56  ✅ standard
'€1,234.56'                  1234.56  ✅ euro symbol, EN thousands
'1.234,56'                   1234.56  ✅ EU decimal format (DE/FR/BG)
'2.13%'                         2.13  ✅ CTR as percentage string
'--'                             0.0  ✅ Meta missing-data marker
''                               0.0  ✅ empty cell
None                             0.0  ✅ NaN / None
'1,000,000'                1000000.0  ✅ million impressions

All cases: ✅ PASS
